# 🤖 LLMs desde Cero: Arquitectura Real con Pesos Aleatorios

## ¿Qué demuestra este notebook?

> **Idea central:** Un LLM es una máquina matemática. Su arquitectura define *cómo* transforma texto en texto. Los *pesos* son los que determinan si esa transformación tiene sentido o no.
>
> Aquí usamos:
> - ✅ El **tokenizador real** de Qwen3 (descarga ~1 MB, no el modelo)
> - 🎲 **Pesos aleatorios** — inicializados con `torch`, sin entrenar
> - ✅ **Prompts reales** en español
> - ✅ **Detokenización real** — el output se convierte de vuelta a texto
>
> El texto generado será **incoherente** — y eso es exactamente el punto.

### Estructura:
1. Instalación y tokenizador real
2. Arquitectura Qwen3 + pesos aleatorios
3. Los hiperparámetros explicados con ejemplos concretos
4. Generación **determinista** con prompts reales
5. Generación **no determinista** con prompts reales

---
## 📦 Sección 1: Instalación y Tokenizador Real

Solo descargamos el **tokenizador** de Qwen3 — no el modelo (que pesa GB). El tokenizador son ~1 MB de archivos de vocabulario.

In [ ]:
# Instalaciones necesarias
# pip install torch transformers
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118  # con CUDA

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device}")

In [ ]:
from transformers import AutoTokenizer

# Descarga SOLO el tokenizador (~1 MB), no los pesos del modelo
# Qwen3 usa el mismo tokenizador en todos sus tamaños
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

print(f"✅ Tokenizador cargado")
print(f"   Vocabulario: {tokenizer.vocab_size:,} tokens")
print(f"   Token EOS: {tokenizer.eos_token!r} (ID: {tokenizer.eos_token_id})")
print(f"   Token PAD: {tokenizer.pad_token!r}")

In [ ]:
# ============================================================
#  ¿Qué hace el tokenizador? — Ejemplos concretos
# ============================================================

ejemplos = [
    "Hola, ¿cómo estás?",
    "¿Cuánto es 1 + 1?",
    "La inteligencia artificial aprende de datos.",
]

print("El tokenizador convierte texto en IDs numéricos (y viceversa):\n")
for texto in ejemplos:
    ids = tokenizer.encode(texto)
    tokens_str = [tokenizer.decode([i]) for i in ids]
    print(f"  Texto  : {texto!r}")
    print(f"  IDs    : {ids}")
    print(f"  Tokens : {tokens_str}")
    print()

---
## 🔧 Sección 2: Arquitectura Qwen3 con Pesos Aleatorios

In [ ]:
# ============================================================
#  Bloques de la arquitectura
# ============================================================

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        return self.fc3(F.silu(self.fc1(x)) * self.fc2(x))


class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype
        if self.qwen3_compatible:
            x = x.to(torch.float32)
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps) * self.scale
        if self.shift is not None:
            norm_x = norm_x + self.shift
        return norm_x.to(input_dtype)


def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[:(head_dim // 2)] / head_dim))
    positions = torch.arange(context_length, dtype=dtype)
    angles = torch.cat([positions.unsqueeze(1) * inv_freq.unsqueeze(0)] * 2, dim=1)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x, cos, sin):
    head_dim = x.shape[-1]
    x1, x2 = x[..., :head_dim//2], x[..., head_dim//2:]
    cos = cos[:x.shape[2], :].unsqueeze(0).unsqueeze(0)
    sin = sin[:x.shape[2], :].unsqueeze(0).unsqueeze(0)
    return (x * cos + torch.cat((-x2, x1), dim=-1) * sin).to(x.dtype)


class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None):
        super().__init__()
        assert num_heads % num_kv_groups == 0
        self.num_heads  = num_heads
        self.group_size = num_heads // num_kv_groups
        self.num_kv_groups = num_kv_groups
        if head_dim is None:
            head_dim = d_in // num_heads
        self.head_dim = head_dim
        self.d_out = num_heads * head_dim
        self.W_query  = nn.Linear(d_in, self.d_out,            bias=False, dtype=dtype)
        self.W_key    = nn.Linear(d_in, num_kv_groups*head_dim, bias=False, dtype=dtype)
        self.W_value  = nn.Linear(d_in, num_kv_groups*head_dim, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(self.d_out, d_in,            bias=False, dtype=dtype)
        self.q_norm = RMSNorm(head_dim, eps=1e-6) if qk_norm else None
        self.k_norm = RMSNorm(head_dim, eps=1e-6) if qk_norm else None

    def forward(self, x, mask, cos, sin):
        b, t, _ = x.shape
        q = self.W_query(x).view(b, t, self.num_heads,     self.head_dim).transpose(1,2)
        k = self.W_key(x).view(b, t,   self.num_kv_groups, self.head_dim).transpose(1,2)
        v = self.W_value(x).view(b, t,  self.num_kv_groups, self.head_dim).transpose(1,2)
        if self.q_norm: q = self.q_norm(q)
        if self.k_norm: k = self.k_norm(k)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
        scores = (q @ k.transpose(2,3)).masked_fill(mask, -torch.inf)
        attn = torch.softmax(scores / self.head_dim**0.5, dim=-1)
        return self.out_proj((attn @ v).transpose(1,2).reshape(b, t, self.d_out))


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att   = GroupedQueryAttention(cfg["emb_dim"], cfg["n_heads"], cfg["n_kv_groups"],
                                           cfg["head_dim"], cfg["qk_norm"], cfg["dtype"])
        self.ff    = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"])
        self.norm2 = RMSNorm(cfg["emb_dim"])

    def forward(self, x, mask, cos, sin):
        x = x + self.att(self.norm1(x), mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb    = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        self.trf_blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])
        head_dim = cfg["head_dim"] or cfg["emb_dim"] // cfg["n_heads"]
        cos, sin = compute_rope_params(head_dim, cfg["rope_base"], cfg["context_length"])
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg

    def forward(self, in_idx):
        x = self.tok_emb(in_idx)
        t = x.shape[1]
        mask = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        return self.out_head(self.final_norm(x).to(self.cfg["dtype"]))

print("✅ Arquitectura definida.")

---
## 📐 Sección 3: Hiperparámetros — ¿Qué significa cada uno?

Estos valores definen la **forma** del modelo. No se aprenden, se eligen antes de construirlo.
Usaremos el modelo real **Qwen3-0.6B** para que los números sean reales.

In [ ]:
# ============================================================
#  Hiperparámetros de Qwen3-0.6B — valores REALES del modelo
# ============================================================
QWEN3_CONFIG = {
    # ----------------------------------------------------------
    # vocab_size: ¿Cuántos tokens distintos conoce el modelo?
    # El tokenizador descompone texto en piezas (subpalabras).
    # Por ejemplo: "increíble" -> ["inc", "reí", "ble"] -> [3 IDs]
    # Un vocab mayor = más tokens únicos, menos fragmentación.
    "vocab_size": 151_936,

    # ----------------------------------------------------------
    # context_length: ¿Cuántos tokens puede "leer" a la vez?
    # Es la "ventana de memoria" del modelo.
    # Si el contexto es 40960 tokens y escribes más, los más
    # antiguos desaparecen — el modelo literalmente no los ve.
    "context_length": 40_960,

    # ----------------------------------------------------------
    # emb_dim: Tamaño del vector que representa cada token.
    # Cada token existe como un punto en un espacio de 1024 dims.
    # Tokens con significados similares quedan cerca en ese espacio.
    "emb_dim": 1024,

    # ----------------------------------------------------------
    # n_heads: Cabezas de atención. Cada una aprende a fijarse
    # en relaciones distintas (sintaxis, coreferencía, semántica...).
    "n_heads": 16,

    # ----------------------------------------------------------
    # n_layers: Bloques Transformer apilados.
    # Las primeras capas capturan patrones locales (letras, palabras),
    # las últimas capturan conceptos abstractos (ideas, razonamiento).
    "n_layers": 28,

    # ----------------------------------------------------------
    # hidden_dim: Tamaño interno de la red densa (FeedForward)
    # dentro de cada bloque. Suele ser ~3x emb_dim.
    "hidden_dim": 3072,

    # ----------------------------------------------------------
    # head_dim: Dimensión de cada cabeza en GQA.
    # Qwen3 lo fija en 128 para todos los tamaños del modelo.
    "head_dim": 128,

    # ----------------------------------------------------------
    # n_kv_groups: Grupos en Grouped Query Attention.
    # Multi-Head clásico: n_kv_groups == n_heads (más parámetros)
    # GQA: n_kv_groups < n_heads (Keys y Values compartidos)
    # Aquí: 8 grupos para 16 cabezas -> cada par K/V lo usan 2 cabezas
    "n_kv_groups": 8,

    # ----------------------------------------------------------
    # qk_norm: Normalizar queries y keys antes de la atención.
    # Estabiliza el entrenamiento evitando explosión de gradientes.
    "qk_norm": True,

    # ----------------------------------------------------------
    # rope_base: Base de Rotary Position Embeddings.
    # Valor mayor = el modelo puede distinguir posiciones más lejanas.
    # GPT-2 usaba embeddings posicionales fijos. RoPE es más flexible.
    "rope_base": 1_000_000.0,

    # ----------------------------------------------------------
    # dtype: Precisión numérica.
    # bfloat16 = 2 bytes/parámetro (mitad de RAM que float32)
    # El modelo usa bfloat16 en producción. Aquí float32 para compatibilidad.
    "dtype": torch.float32,
}

print("📊 Hiperparámetros de Qwen3-0.6B:")
print(f"   vocab_size      : {QWEN3_CONFIG['vocab_size']:>10,} tokens")
print(f"   context_length  : {QWEN3_CONFIG['context_length']:>10,} tokens (~31.000 palabras)")
print(f"   emb_dim         : {QWEN3_CONFIG['emb_dim']:>10,} dimensiones por token")
print(f"   n_layers        : {QWEN3_CONFIG['n_layers']:>10,} bloques transformer")
print(f"   n_heads         : {QWEN3_CONFIG['n_heads']:>10,} cabezas de atención")
print(f"   n_kv_groups     : {QWEN3_CONFIG['n_kv_groups']:>10,} grupos K/V")
print(f"   hidden_dim      : {QWEN3_CONFIG['hidden_dim']:>10,}")
print(f"   head_dim        : {QWEN3_CONFIG['head_dim']:>10,}")

In [ ]:
# ============================================================
#  Ejemplos concretos de cada hiperparámetro con el tokenizador real
# ============================================================

print("=" * 65)
print("EJEMPLO: vocab_size = 151.936")
print("=" * 65)
texto = "Hola, ¿cómo estás?"
ids   = tokenizer.encode(texto)
print(f"  El tokenizador convierte: {texto!r}")
print(f"  En IDs dentro de [0, 151935]: {ids}")
print(f"  Cada ID es un índice en la tabla de embeddings de tamaño {QWEN3_CONFIG['vocab_size']:,}")

print()
print("=" * 65)
print("EJEMPLO: context_length = 40.960")
print("=" * 65)
texto_largo = "La IA " * 500  # 500 repeticiones ~ varios miles de tokens
ids_largo   = tokenizer.encode(texto_largo)
contexto    = QWEN3_CONFIG["context_length"]
print(f"  Texto artificial de {len(ids_largo):,} tokens")
print(f"  context_length = {contexto:,}")
if len(ids_largo) > contexto:
    print(f"  -> Se truncaría: el modelo solo ve los últimos {contexto:,} tokens")
else:
    print(f"  -> Cabe completo en el contexto")

print()
print("=" * 65)
print("EJEMPLO: emb_dim = 1024")
print("=" * 65)
emb_table = nn.Embedding(QWEN3_CONFIG["vocab_size"], QWEN3_CONFIG["emb_dim"])
token_id  = torch.tensor([ids[0]])
vector    = emb_table(token_id).detach()
print(f"  El token {ids[0]} ({tokenizer.decode([ids[0]])!r}) -> vector de shape {vector.shape}")
print(f"  Primeros 8 valores: {vector[0, :8].numpy().round(3).tolist()}")
print(f"  Con pesos ALEATORIOS estos valores no tienen significado.")
print(f"  Tras entrenamiento, tokens similares quedan cerca en el espacio de 1024 dims.")

print()
print("=" * 65)
print("EJEMPLO: n_heads=16, n_kv_groups=8 (GQA)")
print("=" * 65)
print(f"  Multi-Head Attention clásico: 16 cabezas Q, 16 K, 16 V")
print(f"  Grouped Query Attention:      16 cabezas Q, 8 K,  8 V")
print(f"  -> Cada par K/V es compartido por 2 cabezas Q")
params_mha = 3 * 16 * QWEN3_CONFIG['head_dim'] * QWEN3_CONFIG['emb_dim']
params_gqa = (16 + 8 + 8) * QWEN3_CONFIG['head_dim'] * QWEN3_CONFIG['emb_dim']
print(f"  Parámetros QKV en MHA: {params_mha:,}")
print(f"  Parámetros QKV en GQA: {params_gqa:,}  ({(1-params_gqa/params_mha)*100:.0f}% menos)")

In [ ]:
# ============================================================
#  Inicializar el modelo con PESOS ALEATORIOS
# ============================================================
# ⚠️ Esto NO descarga pesos de internet.
#    PyTorch inicializa todo con distribución normal aleatoria.
#    La arquitectura es idéntica a Qwen3-0.6B real.

torch.manual_seed(42)

model = Qwen3Model(QWEN3_CONFIG)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
size_gb_f32  = total_params * 4 / 1024**3
size_gb_bf16 = total_params * 2 / 1024**3

print("🎲 Modelo Qwen3-0.6B inicializado con PESOS ALEATORIOS")
print(f"   Parámetros: {total_params:,}")
print(f"   RAM (float32) : {size_gb_f32:.2f} GB")
print(f"   RAM (bfloat16): {size_gb_bf16:.2f} GB  <- lo que usa el modelo real")
print()
print("⚠️  Los pesos son números aleatorios. La arquitectura es real.")
print("   El modelo PUEDE generar texto. Solo que será incoherente.")

---
## 🔒 Sección 4: Generación DETERMINISTA (Greedy / Argmax)

**Determinista** = misma entrada, misma salida, siempre.

En cada paso elegimos el token con **mayor logit** (argmax). No hay aleatoriedad.

```
texto real → tokenizer.encode() → IDs → modelo → logits → argmax → ID → tokenizer.decode() → texto
```

> 🔁 Ejecuta las celdas las veces que quieras — el resultado nunca cambia.

In [ ]:
def generar_determinista(model, tokenizer, prompt, max_new_tokens=20, verbose=True):
    """
    Generación greedy (argmax): en cada paso elige el token más probable.
    100% determinista — sin aleatoriedad.
    """
    model.eval()

    # 1. Tokenizar el prompt con el tokenizador REAL
    input_ids = tokenizer.encode(prompt)
    token_ids = torch.tensor([input_ids])

    if verbose:
        print(f"  Prompt            : {prompt!r}")
        print(f"  Tokens de entrada : {input_ids}")
        print(f"  ({len(input_ids)} tokens)")
        print()
        print(f"  {'Paso':>4}  {'Token ID':>10}  {'Token (texto)':>20}  {'Prob max':>10}")
        print("  " + "-" * 52)

    tokens_nuevos = []

    with torch.no_grad():
        for paso in range(max_new_tokens):
            # 2. Forward pass
            logits = model(token_ids)          # shape: (1, seq_len, vocab_size)
            logits_ultimo = logits[0, -1, :]   # logits solo del último token

            # 3. Obtener probabilidades
            probs = torch.softmax(logits_ultimo, dim=-1)

            # 4. ARGMAX — token con mayor probabilidad (determinista)
            next_id = torch.argmax(probs).item()
            prob_max = probs[next_id].item()

            # 5. Detokenizar el token generado
            token_texto = tokenizer.decode([next_id])

            if verbose:
                print(f"  {paso+1:>4}  {next_id:>10}  {token_texto!r:>20}  {prob_max:>10.4f}")

            tokens_nuevos.append(next_id)
            token_ids = torch.cat([token_ids, torch.tensor([[next_id]])], dim=1)

            # Parar si se genera el token de fin de secuencia
            if next_id == tokenizer.eos_token_id:
                if verbose:
                    print(f"\n  [EOS detectado en paso {paso+1}]")
                break

    # 6. Detokenizar toda la respuesta generada
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)

    return texto_generado, tokens_nuevos

In [ ]:
# ============================================================
#  PROMPT 1: "Hola, ¿cómo estás?"
# ============================================================

print("=" * 65)
print("PROMPT: 'Hola, ¿cómo estás?'")
print("Estrategia: ARGMAX (determinista)")
print("=" * 65)
print()

texto_out, ids_out = generar_determinista(
    model, tokenizer,
    prompt="Hola, ¿cómo estás?",
    max_new_tokens=20
)

print()
print("=" * 65)
print(f"TEXTO GENERADO: {texto_out!r}")
print("=" * 65)
print()
print("💡 El texto es incoherente porque los pesos son aleatorios.")
print("   Pero el proceso es IDÉNTICO al de cualquier LLM real:")
print("   texto → IDs → modelo → logits → argmax → IDs → texto")

In [ ]:
# ============================================================
#  PROMPT 2: "¿Cuánto es 1 + 1?"
# ============================================================

print("=" * 65)
print("PROMPT: '¿Cuánto es 1 + 1?'")
print("Estrategia: ARGMAX (determinista)")
print("=" * 65)
print()

texto_out2, ids_out2 = generar_determinista(
    model, tokenizer,
    prompt="¿Cuánto es 1 + 1?",
    max_new_tokens=20
)

print()
print("=" * 65)
print(f"TEXTO GENERADO: {texto_out2!r}")
print("=" * 65)

In [ ]:
# ============================================================
#  VERIFICACIÓN DE DETERMINISMO
#  Mismo prompt, 3 ejecuciones — deben ser idénticas
# ============================================================

print("Verificando determinismo: 3 ejecuciones del mismo prompt...\n")

resultados = []
for i in range(3):
    texto, _ = generar_determinista(
        model, tokenizer,
        prompt="Hola, ¿cómo estás?",
        max_new_tokens=10,
        verbose=False
    )
    resultados.append(texto)
    print(f"  Ejecución {i+1}: {texto!r}")

todos_iguales = all(r == resultados[0] for r in resultados)
print(f"\n{'✅ CONFIRMADO: todas las ejecuciones son idénticas' if todos_iguales else '❌ Resultados distintos'}")

---
## 🎲 Sección 5: Generación NO DETERMINISTA (Sampling)

Aquí introducimos **aleatoriedad** en la selección del siguiente token. En vez de elegir siempre el más probable, *sampleamos* de la distribución.

Dos parámetros clave:

| Parámetro | Qué hace | Bajo | Alto |
|-----------|----------|------|------|
| **`temperature`** | Aplana/afila la distribución de probabilidad | Conservador, predecible | Creativo, caótico |
| **`top_k`** | Solo considera los K tokens más probables | Poca variedad | Más variedad |

> 🔁 Ejecuta las mismas celdas varias veces — el resultado cambia cada vez.

In [ ]:
def generar_no_determinista(model, tokenizer, prompt, max_new_tokens=20,
                             temperature=1.0, top_k=50, seed=None, verbose=True):
    """
    Generación con sampling: elige el siguiente token al azar
    ponderado por probabilidades (con temperature y top-k).
    
    Args:
        temperature : > 1.0 más creativo, < 1.0 más conservador
        top_k       : solo considera los K tokens más probables
        seed        : None = diferente cada vez; int = reproducible
    """
    model.eval()
    if seed is not None:
        torch.manual_seed(seed)

    input_ids = tokenizer.encode(prompt)
    token_ids = torch.tensor([input_ids])

    if verbose:
        print(f"  Prompt        : {prompt!r}")
        print(f"  temperature   : {temperature}")
        print(f"  top_k         : {top_k}")
        print()
        print(f"  {'Paso':>4}  {'Token ID':>10}  {'Token (texto)':>20}  {'Prob elegida':>13}")
        print("  " + "-" * 55)

    tokens_nuevos = []

    with torch.no_grad():
        for paso in range(max_new_tokens):
            logits = model(token_ids)[0, -1, :]

            # Paso 1 — temperature: divide los logits
            # < 1.0 -> diferencias se amplifican (ganador más dominante)
            # > 1.0 -> diferencias se reducen (distribución más plana)
            logits_scaled = logits / temperature

            # Paso 2 — top-k: descarta todos excepto los K mejores
            if top_k is not None and top_k > 0:
                valores_top, _ = torch.topk(logits_scaled, k=min(top_k, logits_scaled.shape[-1]))
                umbral = valores_top[-1]
                logits_scaled = logits_scaled.masked_fill(logits_scaled < umbral, -float('inf'))

            # Paso 3 — softmax: convertir en probabilidades
            probs = torch.softmax(logits_scaled, dim=-1)

            # Paso 4 — SAMPLE: aquí está la aleatoriedad
            next_id = torch.multinomial(probs, num_samples=1).item()
            prob_elegida = probs[next_id].item()

            token_texto = tokenizer.decode([next_id])

            if verbose:
                print(f"  {paso+1:>4}  {next_id:>10}  {token_texto!r:>20}  {prob_elegida:>13.4f}")

            tokens_nuevos.append(next_id)
            token_ids = torch.cat([token_ids, torch.tensor([[next_id]])], dim=1)

            if next_id == tokenizer.eos_token_id:
                if verbose:
                    print(f"\n  [EOS detectado en paso {paso+1}]")
                break

    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado, tokens_nuevos

In [ ]:
# ============================================================
#  PROMPT 1: "Hola, ¿cómo estás?"
#  Ejecutar varias veces — cada vez sale diferente
# ============================================================

print("=" * 65)
print("PROMPT: 'Hola, ¿cómo estás?'")
print("Estrategia: SAMPLING (no determinista)")
print("=" * 65)
print()

texto_nd, _ = generar_no_determinista(
    model, tokenizer,
    prompt="Hola, ¿cómo estás?",
    max_new_tokens=20,
    temperature=1.0,
    top_k=50,
    seed=None     # ← None = diferente cada vez que ejecutas
)

print()
print("=" * 65)
print(f"TEXTO GENERADO: {texto_nd!r}")
print("=" * 65)
print()
print("🔁 Vuelve a ejecutar esta celda — el resultado cambiará")

In [ ]:
# ============================================================
#  PROMPT 2: "¿Cuánto es 1 + 1?"
#  Comparando distintas temperaturas
# ============================================================

print("=" * 65)
print("PROMPT: '¿Cuánto es 1 + 1?'")
print("Efecto de la TEMPERATURE")
print("=" * 65)

configs_temp = [
    (0.3, "CONSERVADOR  — poca variedad, repite tokens dominantes"),
    (1.0, "BALANCEADO   — distribución original del modelo"),
    (2.0, "CAÓTICO      — tokens poco probables aparecen más"),
]

for temp, desc in configs_temp:
    print(f"\n🌡️  temperature={temp}  {desc}")
    texto, _ = generar_no_determinista(
        model, tokenizer,
        prompt="¿Cuánto es 1 + 1?",
        max_new_tokens=15,
        temperature=temp,
        top_k=50,
        seed=99,     # misma semilla para comparar temperaturas
        verbose=False
    )
    print(f"   Generado: {texto!r}")

In [ ]:
# ============================================================
#  Efecto de top_k
# ============================================================

print("=" * 65)
print("PROMPT: 'La inteligencia artificial'")
print("Efecto de TOP_K")
print("=" * 65)

configs_k = [
    (1,   "top_k=1   — equivale a argmax (determinista con seed)"),
    (10,  "top_k=10  — solo los 10 tokens más probables"),
    (200, "top_k=200 — amplia variedad de candidatos"),
]

for k, desc in configs_k:
    print(f"\n🎯 {desc}")
    texto, _ = generar_no_determinista(
        model, tokenizer,
        prompt="La inteligencia artificial",
        max_new_tokens=15,
        temperature=1.0,
        top_k=k,
        seed=7,
        verbose=False
    )
    print(f"   Generado: {texto!r}")

In [ ]:
# ============================================================
#  COMPARACIÓN FINAL
#  Mismo prompt, misma arquitectura, mismos pesos
#  Determinista vs No Determinista
# ============================================================

PROMPT = "Explícame qué es una red neuronal"
N = 4

print("=" * 65)
print(f"PROMPT: {PROMPT!r}")
print("=" * 65)

print(f"\n🔒 DETERMINISTA (argmax) — {N} ejecuciones:")
for i in range(N):
    texto, _ = generar_determinista(model, tokenizer, PROMPT, max_new_tokens=10, verbose=False)
    print(f"  [{i+1}] {texto!r}")

print(f"\n🎲 NO DETERMINISTA (sampling, temp=0.9, top_k=40) — {N} ejecuciones:")
for i in range(N):
    texto, _ = generar_no_determinista(
        model, tokenizer, PROMPT,
        max_new_tokens=10, temperature=0.9, top_k=40, seed=None, verbose=False
    )
    print(f"  [{i+1}] {texto!r}")

print()
print("=" * 65)
print("CONCLUSIÓN")
print("=" * 65)
print("""
  ✅ El tokenizador real convirtió el prompt en IDs reales de Qwen3
  ✅ El modelo ejecutó la arquitectura completa (GQA, RoPE, SwiGLU)
  ✅ Los logits se convirtieron de vuelta en texto real (detokenización)
  ❌ El texto es incoherente porque los pesos son ALEATORIOS

  La diferencia entre este output y el de Qwen3 real no es la
  arquitectura ni el proceso de generación — son los pesos.
  600 millones de parámetros ajustados sobre petabytes de texto.
""")